In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 3: Two-Tower Retrieval at Scale

### Business Context
Retrieve top-500 relevant items from a catalog of 5M in <20ms per request. Current matrix factorization scores all items at serving time — 800ms latency, not scalable.

### Architecture
```
User Tower                 Item Tower
user_id emb               item_id emb
history emb (mean pool)   category emb
user attributes           item attributes
      |                         |
      MLP                       MLP
      |                         |
   u ∈ R^d              v ∈ R^d (pre-computed, indexed)
      |                         |
      +--------dot(u,v)----------+
                  |
           Relevance score

Serving: encode user (1 forward pass) → ANN query over item index
```

### Key Concepts

**1. In-batch Negatives (InfoNCE loss):**  
For each positive pair (u_i, v_i), treat all other items in the batch as negatives.  
Efficient — no separate negative mining. With batch size B, each sample gets B-1 negatives.
```
logits = u @ V.T          # (B, B) similarity matrix
labels = eye(B)           # diagonal = positive pairs
loss   = cross_entropy(logits / temperature, labels)
```

**2. Hard Negative Mining:**  
Random negatives are too easy (clearly irrelevant). Hard negatives = items the model ranks highly but user didn't interact with. Improves embedding separation. Typical: 1 positive : 1 random : 1 hard negative.

**3. ANN Index Trade-offs:**
| Index | Build Time | Memory | Recall@100 | Query Latency |
|-------|-----------|--------|------------|---------------|
| FAISS IVF | Fast | Low | ~95% | ~5ms |
| FAISS HNSW | Slow | High | ~99% | ~2ms |
| ScaNN | Medium | Low | ~97% | ~3ms |

HNSW: best recall/latency, graph-based. IVF: clusters items into K centroids, searches nprobe nearest. ScaNN: optimized for inner product (Google's production system).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Tower(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  # L2-normalize for cosine similarity

class TwoTowerModel(nn.Module):
    def __init__(self, user_dim=32, item_dim=32, embed_dim=64, temperature=0.07):
        super().__init__()
        self.user_tower = Tower(user_dim, 128, embed_dim)
        self.item_tower = Tower(item_dim, 128, embed_dim)
        self.temperature = temperature

    def forward(self, user_features, item_features):
        u = self.user_tower(user_features)    # (B, d)
        v = self.item_tower(item_features)    # (B, d)
        return u, v

    def infonce_loss(self, u, v):
        """
        In-batch negative loss (InfoNCE).
        u: (B, d) user embeddings
        v: (B, d) item embeddings — row i is the positive item for user i
        """
        B = u.shape[0]
        logits = (u @ v.T) / self.temperature   # (B, B)
        labels = torch.arange(B)                # diagonal = positive pairs
        loss = F.cross_entropy(logits, labels)
        return loss

# Training step simulation
model = TwoTowerModel(user_dim=16, item_dim=16, embed_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

B = 64   # batch size
losses = []
for step in range(100):
    user_feats = torch.randn(B, 16)
    item_feats = torch.randn(B, 16)  # row i = positive item for user i
    u, v = model(user_feats, item_feats)
    loss = model.infonce_loss(u, v)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f'Training loss: step 0={losses[0]:.4f} -> step 99={losses[-1]:.4f}')
print(f'Expected minimum (random negatives, B=64): {np.log(B):.4f}')

# Serving: pre-compute item embeddings and do ANN lookup
n_items = 1000
all_item_feats = torch.randn(n_items, 16)
with torch.no_grad():
    item_index = model.item_tower(all_item_feats).numpy()   # (n_items, 32)

query_user = torch.randn(1, 16)
with torch.no_grad():
    user_emb = model.user_tower(query_user).numpy()         # (1, 32)

# Exact NN (in practice, use FAISS)
scores = user_emb @ item_index.T                           # (1, n_items)
top_k  = np.argsort(-scores[0])[:10]
print(f'\nTop-10 retrieved item indices: {top_k}')
print(f'Top-10 scores: {scores[0][top_k].round(3)}')

### L6 Interview Q&A

**Q: Why can't you use a cross-encoder (joint user-item model) for retrieval?**  
A: Cross-encoders process the user and item jointly, so you can't pre-compute item representations. At serving time you'd need one forward pass per item — 5M forward passes is not feasible at <20ms. Two-tower decouples the encodings, pre-computes all item embeddings offline, and only requires one user encoding + ANN lookup at serving time.

**Q: What's the trade-off between temperature in InfoNCE loss?**  
A: Low temperature (e.g. 0.07) sharpens the distribution — model focuses on the hardest negatives in the batch, learns finer-grained distinctions. High temperature makes the loss smoother, easier to optimize but less discriminative. Too low → training instability (gradient exploding). Typical range: 0.05–0.1. Temperature is often a tunable hyperparameter.

**Q: How do you evaluate a retrieval model offline?**  
A: Recall@K — fraction of true positive items found in the top-K retrieved. Target: Recall@500 ≥ 92% so the downstream ranker has sufficient signal. Also check coverage (fraction of catalog ever retrieved) to detect popularity bias in the index.

### Common Pitfalls
- Not L2-normalizing embeddings — cosine similarity requires normalized vectors
- Using batch size too small — InfoNCE needs enough negatives (B ≥ 256 typically)
- Forgetting that item index must be rebuilt when item catalog changes